# Notebook 4: Agentic RAG Introduction

In this notebook you will learn how to classify a user search and have specialize agents called depending on the user's question. This notebook does not use the Azure AI Search index to create a full solution for you - you will get to do that in the hands on exercise.

> NOTE: This notebook is heavily inspired by the [Simple Handoff sample](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/orchestrations/handoff_simple.py) in the agent-framework source tree.

## Learning Objectives
- Learn about the Handoff pattern
- Implement a classifier to recognize the user's intent and route to a specialized agent
- Implement agents to respond to specific search types


### Setup the Module Imports

In [14]:
import os

from collections.abc import AsyncIterable
from typing import Annotated, cast

from agent_framework import (
    Agent,
    AgentResponse,
    Message,
    WorkflowEvent,
    WorkflowRunState,
    tool
)
from agent_framework.openai import OpenAIChatClient
from agent_framework.orchestrations import HandoffAgentUserRequest, HandoffBuilder
from azure.identity import DefaultAzureCredential

Define AI functions you'll use for testing the specific search type in this notebook.

> NOTE: these are where the actual implementation goes for doing specific search types in the hands on notebook

In [2]:
@tool(approval_mode="never_require")
def yes_or_no_search(user_question: Annotated[str, "User question to answer with yes or no"]) -> str:
    """Answers yes or no questions."""
    return f"Question: {user_question}\nAnswer: NO"

@tool(approval_mode="never_require")
def count_search(user_question: Annotated[str, "User question requiring counting items"]) -> str:
    """Answers questinos that required counting."""
    return f"Question: {user_question}\nAnswer: 42"

@tool(approval_mode="never_require")
def semantic_search(user_question: Annotated[str, "User question requiring semantic search"]) -> str:
    """Answers simple question that requires semantic search."""
    return f"Question: {user_question}\nAnswer: They all the gray or metallic in color."

Create a prompt for the classifier to be able to determine how to route the user's question.

In [3]:
CLASSIFIER_AGENT_INSTRUCTIONS = """
You are a query classification system for an IT support ticket database. Your task is to route them to specialist search agents based on the user question.

## Database Schema
The database contains IT support tickets with these fields:
- Id: unique identifier
- Subject: ticket subject
- Body: ticket question/description
- Answer: ticket response/solution
- Type: ticket type (e.g., "Incident", "Request", "Problem")
- Queue: department name (e.g., "Human Resources", "IT", "Finance")
- Priority: "high", "medium", or "low"
- Language: ticket language
- Business_Type: business category
- Tags: categorization tags

## Types of Searches Agents can do:

**YES_NO_AGENT**: Simple yes/no questions
   - Keywords: "is", "are", "can", "does", "do", "will", "should"
   - Examples:
     - "Is my account locked?"
     - "Can I access the VPN?"
     - "Does the printer support color printing?"
     - "Will my password expire soon?"

**COUNT_AGENT**: Questions requiring counting items
   - Keywords: "how many", "number of", "count of", "total"
    - Examples:
      - "How many devices are connected to the network?"
      - "What is the total number of open tickets?"
      - "Count of users with admin access"

**SEMANTIC_SEARCH_AGENT**: Queries looking for similar issues or solutions based on meaning
   - Keywords: "how to", "why", "what causes", "solve", "fix", "issue with", "problem with"
   - Examples:
     - "How do I reset my password?"
     - "Why is my VPN not connecting?"
     - "Issues with email synchronization"
     - "Laptop won't turn on"

**NO_SEARCH_NEEDED**: Queries that do not require any search
   - Examples:
      - "Hello I need assistance."
      - "Thank you for your help."
      - "Goodbye."
      - "No further questions."

Consider:
1. The primary intent - what is the user trying to achieve?
2. Whether they need specific data points (analytical) vs similar content (semantic)
3. Whether filters/constraints are present
4. The type of expected response (number, list, specific tickets, similar issues)
5. If no search is needed, respond with NO_SEARCH_NEEDED.

"""

Define a method that will create all the agents and connect the tool functions created earlier to their respective agents.

In [5]:
def create_agents(chat_client: OpenAIChatClient) -> tuple[Agent, Agent, Agent, Agent]:
    """Create and configure the classifier and search specialist agents.

    The classifier agent is responsible for:
    - Receiving all user input first
    - Deciding whether to handle the request directly or hand off to a search specialist
    - Signaling handoff by calling one of the explicit handoff tools exposed to it

    Search specialist agents are invoked only when the classifier agent explicitly hands off to them.
    After a specialist responds, control returns to the classifier agent, which then prompts
    the user for their next message.

    Returns:
        Tuple of (classifier_agent, yes_no_agent, count_agent, semantic_search_agent)
    """
    # Classifier agent: Acts as the search planner and dispatcher
    classifier_agent = chat_client.as_agent(
        instructions=CLASSIFIER_AGENT_INSTRUCTIONS,
        name="classifier_agent",
        require_per_service_call_history_persistence=True,
    )

    # Yes/No search specialist: Handles yes/no questions
    yes_no_agent = chat_client.as_agent(
        instructions="You handle yes/no questions.",
        name="yes_no_agent",
        tools=[yes_or_no_search],
        require_per_service_call_history_persistence=True,
    )

    # Count search specialist: Handles counting questions
    count_agent = chat_client.as_agent(
        instructions="You handle questions that require counting items.",
        name="count_agent",
        tools=[count_search],
        require_per_service_call_history_persistence=True,
    )

    # Semantic search specialist: Handles semantic search questions
    semantic_search_agent = chat_client.as_agent(
        instructions="You handle questions that require semantic search.",
        name="semantic_search_agent",
        tools=[semantic_search],
        require_per_service_call_history_persistence=True,
    )

    return classifier_agent, yes_no_agent, count_agent, semantic_search_agent

Define a utility method that will show us the agent messages

In [7]:
def _print_handoff_agent_user_request(response: AgentResponse) -> None:
    """Display the agent's response messages when requesting user input.

    This will happen when an agent generates a response that doesn't trigger
    a handoff, i.e., the agent is asking the user for more information.

    Args:
        response: The AgentResponse from the agent requesting user input
    """
    if not response.messages:
        raise RuntimeError("Cannot print agent responses: response has no messages.")

    print("\n[Agent is requesting your input...]")

    # Print agent responses
    for message in response.messages:
        if not message.text:
            # Skip messages without text (e.g., tool calls)
            continue
        speaker = message.author_name or message.role
        print(f"- {speaker}: {message.text}")

Define a utility method that writes out the workflow status or writes out the messages.

In [11]:
def _handle_events(events: list[WorkflowEvent]) -> list[WorkflowEvent[HandoffAgentUserRequest]]:
    """Process workflow events and extract any pending user input requests.

    This function inspects each event type and:
    - Prints workflow status changes (IDLE, IDLE_WITH_PENDING_REQUESTS, etc.)
    - Displays final conversation snapshots when workflow completes
    - Prints user input request prompts
    - Collects all request_info events for response handling

    Args:
        events: List of WorkflowEvent to process

    Returns:
        List of WorkflowEvent[HandoffAgentUserRequest] representing pending user input requests
    """
    requests: list[WorkflowEvent[HandoffAgentUserRequest]] = []

    for event in events:
        if event.type == "handoff_sent":
            # handoff_sent event: Indicates a handoff has been initiated
            print(f"\n[Handoff from {event.data.source} to {event.data.target} initiated.]")
        elif event.type == "status" and event.state in {
            WorkflowRunState.IDLE,
            WorkflowRunState.IDLE_WITH_PENDING_REQUESTS,
        }:
            # Status event: Indicates workflow state changes
            print(f"\n[Workflow Status] {event.state}")
        elif event.type == "output":
            # Output event: Contains contents generated by the workflow
            data = event.data
            if isinstance(data, AgentResponse):
                for message in data.messages:
                    if not message.text:
                        # Skip messages without text (e.g., tool calls)
                        continue
                    speaker = message.author_name or message.role
                    print(f"- {speaker}: {message.text}")
            elif event.type == "output":
                # The output of the handoff workflow is a collection of chat messages from all participants
                conversation = cast(list[Message], event.data)
                if isinstance(conversation, list):
                    print("\n=== Final Conversation Snapshot ===")
                    for message in conversation:
                        speaker = message.author_name or message.role
                        print(f"- {speaker}: {message.text or [content.type for content in message.contents]}")
                    print("===================================")
        elif event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            _print_handoff_agent_user_request(event.data.agent_response)
            requests.append(cast(WorkflowEvent[HandoffAgentUserRequest], event))

    return requests

Now all the code is defined, you get to used it.

Create a chat client and all the agents.

In [ ]:
import dotenv
dotenv.load_dotenv()

# Initialize the Azure OpenAI chat client
chat_client = OpenAIChatClient(
    model=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    credential=DefaultAzureCredential()
)

# Create all agents: classifier + specialists
classifier, yes_no, count, semantic_search = create_agents(chat_client)

Create the workflow, using the [HandoffBuilder](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework.handoffbuilder?view=agent-framework-python-latest), setting all the agents as participants and the classifier as the handoff coordinator.

In [19]:
workflow = (
    HandoffBuilder(
        name="agentic_rag_workflow",
        participants=[classifier, yes_no, count, semantic_search],
    )
    .with_start_agent(classifier)
    .build()
)

For demo purposes create a list of some sample questions.

In [20]:
scripted_responses = [
    "How many tickets were logged and Incidents for Human Resources and low priority?", # count search
    "Are there any issues logged for Dell XPS laptops?" # yes/no search
]

Ask and initial question and then the above demo questions to see if it correctly classifies the search types.

In [21]:
initial_message = "What problems are there with Surface devices?" # semantic search
print(f"- User: {initial_message}")

workflow_result = workflow.run(initial_message, stream=True)
pending_requests = _handle_events([event async for event in workflow_result])

 # Process the request/response cycle
# The workflow will continue requesting input until:
# 1. The termination condition is met, OR
# 2. We run out of scripted responses
while pending_requests:
    if not scripted_responses:
        # No more scripted responses; terminate the workflow
        responses = {req.request_id: HandoffAgentUserRequest.terminate() for req in pending_requests}
    else:
        # Get the next scripted response
        user_response = scripted_responses.pop(0)
        print(f"\n- User: {user_response}")

        # Send response(s) to all pending requests
        # In this demo, there's typically one request per cycle, but the API supports multiple
        responses = {
            req.request_id: HandoffAgentUserRequest.create_response(user_response) for req in pending_requests
        }

    # Send responses and get new events
    # We use run(responses=...) to get events from the workflow, allowing us to
    # display agent responses and handle new requests as they arrive
    events = await workflow.run(responses=responses)
    pending_requests = _handle_events(events)

- User: What problems are there with Surface devices?

[Handoff from classifier_agent to semantic_search_agent initiated.]

[Agent is requesting your input...]
- semantic_search_agent: The search did not return relevant information about specific problems with Surface devices. However, common issues reported by users with Surface devices generally include:

- Battery life that may degrade over time
- Screen or touchscreen responsiveness issues
- Overheating under heavy use
- Keyboard or Type Cover connection problems
- Software glitches or driver issues
- Limited port availability for connectivity

If you want detailed information on problems with a specific Surface model, please let me know!

[Workflow Status] WorkflowRunState.IDLE_WITH_PENDING_REQUESTS

- User: How many tickets were logged and Incidents for Human Resources and low priority?
- semantic_search_agent: The search did not return relevant information about the number of tickets or incidents logged for Human Resources with 

As I've pointed out in the other notebooks, the question "How many tickets were logged and Incidents for Human Resources and low priority?" is a hard one to get the system to answer - and this time the classifier correctly identified it as something the count_agent should take care of. That is a start!

Next is the hands on exercise where you get to turn these lessons from the notebooks into an Agentic RAG solution for the IT support search index we've been using.